In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

In [2]:
df = pd.read_csv('datasets/ratings.csv')
df.columns

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')

In [3]:
Y = df.pivot_table(index='movieId', columns='userId', values='rating').fillna(0).to_numpy()
n_m, n_u = Y.shape
n = 14 # nos. feature
tf.random.set_seed(123)
X = tf.Variable(tf.random.normal((n_m, n)))
W = tf.Variable(tf.random.normal((n_u, n)))
b = tf.Variable(tf.zeros((1, n_u)))
R = tf.constant((Y > 0).astype(float), dtype=tf.float32)

print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("R", R.shape)

X (9724, 14)
W (611, 14)
b (1, 611)
R (9724, 611)


In [4]:
def compute_cost(X, W, b, Y, lambda_):
    cost = tf.reduce_sum(R * (X @ tf.transpose(W) + b - Y)**2) / 2.0
    reg_X = (lambda_ / 2.0) * tf.reduce_sum(X ** 2)
    reg_W = (lambda_ / 2.0) * tf.reduce_sum(W ** 2)
    return cost + reg_X + reg_W

In [5]:
def fit_model(X, W, b, Y, lambda_, lr=0.3, n_iter=100):
    optimizer = tf.keras.optimizers.Adam(lr)

    for epoch in tqdm(range(n_iter), desc="Training Model", unit="epoch"):
        with tf.GradientTape() as tape:
            cost_J = compute_cost(X, W, b, Y, lambda_)

        gradients = tape.gradient(cost_J, [W, X, b])
        optimizer.apply_gradients(zip(gradients, [W, X, b]))

        tqdm.write(f"Epoch {epoch+1}: Loss = {cost_J:.4f}",
                   end="\r" if epoch < n_iter - 1 else "\n")

    return (X, W, b)

In [6]:
sum_ratings = tf.cast(tf.reduce_sum(Y, axis=1), tf.float32)
count_ratings = tf.reduce_sum(R, axis=1)

movie_means = sum_ratings / (count_ratings + 1e-8)
movie_means = tf.reshape(movie_means, (-1, 1))
Y_norm = (Y - movie_means) * R
(X, W, b) = fit_model(X, W, b, Y_norm, 0.1, 0.1, 500)

np.savez('recommender_weights.npz', X=X.numpy(), W=W.numpy(), b=b.numpy(), movie_means=movie_means.numpy())
print("Model variables saved successfully as Numpy arrays!!")

Training Model:   0%|          | 0/500 [00:00<?, ?epoch/s]

Epoch 500: Loss = 8827.50499
Model variables saved successfully as Numpy arrays!!
